In [1]:
import os
import time
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "used_phone_price_prediction_1M.csv"

df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "sharmajicoder/used-phone-price-prediction-dataset",
  file_path,
)

c:\Users\cheta\Code\Personal\Data_Science\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
print("Dataset shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

Dataset shape: (1000000, 28)
Missing values: 0
Duplicate rows: 0


In [4]:
# Keep market_demand_score completely out of the model
X = df.drop(columns=["resale_price", "market_demand_score"])
y = df["resale_price"]

numerical_features = X.select_dtypes(include=["number"]).columns
categorical_features = X.select_dtypes(include=["object", "str"]).columns

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=8
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Feature count:", X.shape[1])
print("Market demand included:", "market_demand_score" in X.columns)

Training rows: 800000
Testing rows: 200000
Feature count: 26
Market demand included: False


In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=2), categorical_features)
    ]
)

start = time.time()
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print("Transformed training shape:", X_train_transformed.shape)
print("Preprocessing minutes:", round((time.time() - start) / 60, 2))

Transformed training shape: (800000, 61)
Preprocessing minutes: 0.09


In [6]:
experiments = [
    {"n_estimators": 20, "max_depth": 10, "min_samples_leaf": 5},
    {"n_estimators": 30, "max_depth": 12, "min_samples_leaf": 5},
    {"n_estimators": 40, "max_depth": 15, "min_samples_leaf": 5},
    {"n_estimators": 50, "max_depth": 18, "min_samples_leaf": 5},
]

results = []
trained_models = []

for i, params in enumerate(experiments, start=1):
    print(f"\nTraining experiment {i}/{len(experiments)}: {params}")
    rf = RandomForestRegressor(
        **params,
        min_samples_split=10,
        n_jobs=-1,
        random_state=42
    )

    start = time.time()
    rf.fit(X_train_transformed, y_train)
    train_minutes = (time.time() - start) / 60

    preds = rf.predict(X_test_transformed)
    mae = mean_absolute_error(y_test, preds)
    rmse = mean_squared_error(y_test, preds) ** 0.5
    r2 = r2_score(y_test, preds)
    node_count = sum(tree.tree_.node_count for tree in rf.estimators_)

    # Estimate serialized size without writing a large file to disk.
    import pickle
    estimated_mb = len(pickle.dumps(rf, protocol=pickle.HIGHEST_PROTOCOL)) / (1024 ** 2)

    results.append({
        "experiment": i,
        "n_estimators": params["n_estimators"],
        "max_depth": params["max_depth"],
        "min_samples_leaf": params["min_samples_leaf"],
        "total_nodes": node_count,
        "estimated_uncompressed_MB": round(estimated_mb, 2),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "training_minutes": round(train_minutes, 2)
    })
    trained_models.append(rf)

    print(f"MAE={mae:.2f} | RMSE={rmse:.2f} | R2={r2:.4f} | "
          f"nodes={node_count:,} | estimate={estimated_mb:.1f} MB | "
          f"train={train_minutes:.1f} min")

results_df = pd.DataFrame(results).sort_values("MAE")
results_df


Training experiment 1/4: {'n_estimators': 20, 'max_depth': 10, 'min_samples_leaf': 5}
MAE=5757.06 | RMSE=8091.01 | R2=0.7649 | nodes=40,542 | estimate=2.8 MB | train=1.0 min

Training experiment 2/4: {'n_estimators': 30, 'max_depth': 12, 'min_samples_leaf': 5}
MAE=5518.91 | RMSE=7725.63 | R2=0.7856 | nodes=216,266 | estimate=14.9 MB | train=1.5 min

Training experiment 3/4: {'n_estimators': 40, 'max_depth': 15, 'min_samples_leaf': 5}
MAE=5336.94 | RMSE=7401.80 | R2=0.8032 | nodes=1,222,172 | estimate=83.9 MB | train=2.1 min

Training experiment 4/4: {'n_estimators': 50, 'max_depth': 18, 'min_samples_leaf': 5}
MAE=5280.66 | RMSE=7300.20 | R2=0.8086 | nodes=3,700,586 | estimate=254.1 MB | train=3.1 min


,experiment,n_estimators,max_depth,min_samples_leaf,total_nodes,estimated_uncompressed_MB,MAE,RMSE,R2,training_minutes
3,4,50,18,5,3700586,254.11,5280.664557,7300.198121,0.808591,3.07
2,3,40,15,5,1222172,83.93,5336.938246,7401.796062,0.803226,2.09
1,2,30,12,5,216266,14.86,5518.905707,7725.629218,0.785631,1.48
0,1,20,10,5,40542,2.79,5757.063311,8091.008826,0.764875,0.95


In [7]:
display(results_df[[
    "experiment", "n_estimators", "max_depth", "min_samples_leaf",
    "total_nodes", "estimated_uncompressed_MB", "MAE", "RMSE", "R2",
    "training_minutes"
]])

,experiment,n_estimators,max_depth,min_samples_leaf,total_nodes,estimated_uncompressed_MB,MAE,RMSE,R2,training_minutes
3,4,50,18,5,3700586,254.11,5280.664557,7300.198121,0.808591,3.07
2,3,40,15,5,1222172,83.93,5336.938246,7401.796062,0.803226,2.09
1,2,30,12,5,216266,14.86,5518.905707,7725.629218,0.785631,1.48
0,1,20,10,5,40542,2.79,5757.063311,8091.008826,0.764875,0.95


In [ ]:
# Change this after reviewing results_df.
SELECTED_EXPERIMENT = int(results_df.iloc[0]["experiment"])

selected_row = results_df.loc[results_df["experiment"] == SELECTED_EXPERIMENT].iloc[0]
selected_rf = trained_models[SELECTED_EXPERIMENT - 1]

selected_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", selected_rf)
])

output_path = "model.pkl"
joblib.dump(selected_pipeline, output_path, compress=3)

size_mb = os.path.getsize(output_path) / (1024 ** 2)
print("Selected experiment:", SELECTED_EXPERIMENT)
print("Selected parameters:", {
    "n_estimators": int(selected_row["n_estimators"]),
    "max_depth": int(selected_row["max_depth"]),
    "min_samples_leaf": int(selected_row["min_samples_leaf"])
})
print(f"Compressed pipeline size: {size_mb:.2f} MB")
print("Saved to:", output_path)

Selected experiment: 4
Selected parameters: {'n_estimators': 50, 'max_depth': 18, 'min_samples_leaf': 5}
Compressed pipeline size: 86.08 MB
Saved to: model_rf_small_compressed.pkl


In [9]:
loaded_model = joblib.load(output_path)
sample_predictions = loaded_model.predict(X_test.iloc[:5])
print("First five predictions:", sample_predictions)
print("First five actual values:", y_test.iloc[:5].to_list())
print("Saved model file exists:", os.path.exists(output_path))

First five predictions: [12509.20790832 24910.32885873 11896.26368451 62759.1413378
 10259.86777413]
First five actual values: [14971.37, 30189.85, 13506.16, 47124.84, 5199.53]
Saved model file exists: True


In [11]:
previous_model = joblib.load("model1.pkl")
current_model = joblib.load("model.pkl")

previous_predictions = previous_model.predict(X_test)
current_predictions = current_model.predict(X_test)

print("Previous model predictions:", previous_predictions)
print("Current model predictions:", current_predictions)

Previous model predictions: [12665.9684 23830.4175 11662.3105 ... 42497.5726 11779.92   13951.7418]
Current model predictions: [12509.20790832 24910.32885873 11896.26368451 ... 41099.74086173
 11580.2046267  14312.75617829]


In [14]:
previous_mae = mean_absolute_error(y_test, previous_predictions)
current_mae = mean_absolute_error(y_test, current_predictions)

previous_rmse = mean_squared_error(y_test, previous_predictions) ** 0.5
current_rmse = mean_squared_error(y_test, current_predictions) ** 0.5

previous_mse = mean_squared_error(y_test, previous_predictions)
current_mse = mean_squared_error(y_test, current_predictions)

previous_r2 = r2_score(y_test, previous_predictions)
current_r2 = r2_score(y_test, current_predictions)

results_comparison = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "MSE", "R2"],
    "Previous Model": [previous_mae, previous_rmse, previous_mse, previous_r2],
    "Current Model": [current_mae, current_rmse, current_mse, current_r2]
})

In [15]:
results_comparison

,Metric,Previous Model,Current Model
0,MAE,5.278588e+03,5.280665e+03
1,RMSE,7.305975e+03,7.300198e+03
2,MSE,5.337728e+07,5.329289e+07
3,R2,8.082878e-01,8.085909e-01
